# 阶段一：数据探索性分析（EDA）

**实验目标：**
1. 熟悉数据集的结构、字段含义与数据类型
2. 掌握数据质量评估方法（缺失值统计、异常值初步识别）
3. 完成基础统计描述
4. 初步数据可视化

In [ ]:
# 环境初始化
import sys
sys.path.append('..')

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 设置数据目录
DATA_DIR = Path('..') / 'data'
FIG_DIR = Path('..') / 'figures' / 'eda'
FIG_DIR.mkdir(parents=True, exist_ok=True)

print('✓ 环境初始化完成')

## 1. 数据加载与结构检查

In [ ]:
# 读取数据
house = pd.read_csv(DATA_DIR / 'raw' / 'hz_house.csv', encoding='utf-8')

print('=' * 50)
print('数据规模')
print('=' * 50)
print(f'行数：{house.shape[0]}')
print(f'列数：{house.shape[1]}')
print()

# 查看前5行数据
print('=' * 50)
print('前5行数据预览')
print('=' * 50)
display(house.head())

In [ ]:
# 查看数据基本信息
print('=' * 50)
print('数据基本信息')
print('=' * 50)
house.info()
print()

# 查看数据类型分布
print('=' * 50)
print('数据类型分布')
print('=' * 50)
print(house.dtypes.value_counts())

In [ ]:
# 查看所有列名
print('=' * 50)
print('所有列名')
print('=' * 50)
for i, col in enumerate(house.columns):
    print(f'  {i+1:2d}. {col} ({house[col].dtype})')

## 2. 数据质量评估

In [ ]:
# 缺失值统计
print('=' * 50)
print('缺失值统计')
print('=' * 50)
missing_count = house.isnull().sum()
missing_pct = (house.isnull().sum() / len(house) * 100).round(2)
missing_df = pd.DataFrame({
    '缺失数量': missing_count,
    '缺失比例(%)': missing_pct
})
missing_df = missing_df[missing_df['缺失数量'] > 0].sort_values('缺失数量', ascending=False)
print(missing_df)
print(f'\n共有 {len(missing_df)} 个字段存在缺失值')

In [ ]:
# 缺失值可视化
missing = house.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

plt.figure(figsize=(10, 4))
missing.plot(kind='bar', color='coral')
plt.title('各字段缺失值统计', fontsize=14)
plt.xlabel('字段名')
plt.ylabel('缺失数量')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(FIG_DIR / 'missing_values.png', dpi=300)
plt.show()

## 3. 数值特征统计描述

In [ ]:
# 数值特征详细统计
print('=' * 50)
print('数值特征详细统计')
print('=' * 50)
numeric_cols = house.select_dtypes(include=[np.number]).columns.tolist()
numeric_stats = house[numeric_cols].describe().T
numeric_stats['中位数'] = house[numeric_cols].median()
numeric_stats['偏度'] = house[numeric_cols].skew()
numeric_stats['峰度'] = house[numeric_cols].kurt()
display(numeric_stats)

In [ ]:
# 类别特征分布
print('=' * 50)
print('类别特征分布')
print('=' * 50)
category_cols = house.select_dtypes(include=['object']).columns.tolist()
for col in category_cols:
    print(f'\n--- {col} ---')
    vc = house[col].value_counts()
    print(f'唯一值数量：{house[col].nunique()}')
    print('前5个类别分布：')
    print(vc.head())

## 4. 数据分布可视化

In [ ]:
# 目标变量分布可视化
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# 单价分布
axes[0, 0].hist(house['单价（元/平米）'].dropna(), bins=50, color='steelblue', edgecolor='white', alpha=0.8)
axes[0, 0].set_title('单价分布')
axes[0, 0].set_xlabel('单价（元/平米）')
axes[0, 0].set_ylabel('频数')

# 总价分布
axes[0, 1].hist(house['总价（万元）'].dropna(), bins=50, color='coral', edgecolor='white', alpha=0.8)
axes[0, 1].set_title('总价分布')
axes[0, 1].set_xlabel('总价（万元）')
axes[0, 1].set_ylabel('频数')

# 建筑面积分布
axes[0, 2].hist(house['建筑面积'].dropna(), bins=50, color='green', edgecolor='white', alpha=0.8)
axes[0, 2].set_title('建筑面积分布')
axes[0, 2].set_xlabel('面积（平方米）')
axes[0, 2].set_ylabel('频数')

# 楼层分布
if '楼层' in house.columns:
    floor_counts = house['楼层'].value_counts()
    floor_counts.plot(kind='bar', ax=axes[1, 0], color='teal')
    axes[1, 0].set_title('楼层分布')
    axes[1, 0].set_xlabel('楼层')
    axes[1, 0].set_ylabel('数量')

# 朝向分布
if '朝向' in house.columns:
    orient_counts = house['朝向'].value_counts().head(8)
    orient_counts.plot(kind='bar', ax=axes[1, 1], color='orange')
    axes[1, 1].set_title('朝向分布（前8）')
    axes[1, 1].set_xlabel('朝向')
    axes[1, 1].set_ylabel('数量')
    axes[1, 1].tick_params(axis='x', rotation=45)

# 户型分布
if '户型' in house.columns:
    layout_counts = house['户型'].value_counts().head(8)
    layout_counts.plot(kind='bar', ax=axes[1, 2], color='purple')
    axes[1, 2].set_title('户型分布（前8）')
    axes[1, 2].set_xlabel('户型')
    axes[1, 2].set_ylabel('数量')
    axes[1, 2].tick_params(axis='x', rotation=45)

plt.suptitle('杭州学区房数据分布总览', fontsize=16)
plt.tight_layout()
plt.savefig(FIG_DIR / 'data_distribution.png', dpi=300)
plt.show()

## 5. 相关性分析

In [ ]:
# 相关性热力图
plt.figure(figsize=(12, 8))
numeric_data = house.select_dtypes(include=[np.number])
corr_matrix = numeric_data.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5)
plt.title('数值特征相关性热力图', fontsize=14)
plt.tight_layout()
plt.savefig(FIG_DIR / 'correlation_heatmap.png', dpi=300)
plt.show()

# 与单价相关性最高的特征
corr_with_target = corr_matrix['单价（元/平米）'].drop('单价（元/平米）').sort_values(ascending=False)
print('与单价相关性最高的前10个特征：')
print(corr_with_target.head(10))

## 6. 区域分析

In [ ]:
# 各区域二手房数量
area_count = house['所在区县'].value_counts()

plt.figure(figsize=(12, 5))
bars = plt.bar(area_count.index, area_count.values, color='steelblue', edgecolor='black')
plt.xlabel('区域', fontsize=12)
plt.ylabel('数量（套）', fontsize=12)
plt.title('杭州各区域二手房数量', fontsize=14)
plt.grid(axis='y', alpha=0.3)

for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{int(height)}', ha='center', va='bottom')

plt.tight_layout()
plt.savefig(FIG_DIR / 'area_count.png', dpi=300)
plt.show()

In [ ]:
# 各区域房价箱线图
plt.figure(figsize=(14, 6))
district_order = house.groupby('所在区县')['单价（元/平米）'].median().sort_values(ascending=False).index
sns.boxplot(x='所在区县', y='单价（元/平米）', data=house, order=district_order, palette='Set2')
plt.title('不同区县房价分布（按中位数排序）', fontsize=14)
plt.xlabel('区县')
plt.ylabel('单价（元/平米）')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(FIG_DIR / 'area_price_boxplot.png', dpi=300)
plt.show()

print('各区县房价中位数：')
print(house.groupby('所在区县')['单价（元/平米）'].median().sort_values(ascending=False))

## 阶段一总结

**数据概况：**
- 数据集包含约3000+条杭州学区房记录
- 包含建筑面积、户型、楼层、朝向、建筑年代、学校信息等多个特征
- 目标变量为`单价（元/平米）`，属于回归任务

**数据质量：**
- 存在少量缺失值（朝向、建筑年代、建筑面积等）
- 需要进行缺失值处理和特征工程

**初步发现：**
- 单价分布呈右偏分布
- 不同区域房价差异显著
- 南向和南北通透的房源占比较高